# vLLM Stream Test

In [1]:
from openai import OpenAI
import time
import re
import requests
import json
from datetime import datetime

# ---- Settings ----
LLM_BASE_URL = "http://localhost:8001/v1"
# LLM_MODEL = "qwen2.5-vl-7b-instruct"
LLM_MODEL = "Qwen2.5-VL-7B-Instruct-NVFP4"

EMBED_BASE_URL = "http://localhost:8002"
EMBED_MODEL = "Qwen3-VL-Embedding-2B"

RERANK_BASE_URL = "http://localhost:8003"
RERANK_MODEL = "Qwen3-VL-Reranker-2B"

SAMPLE_IMAGE_URL = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS6d87zy2l97Gbuz1xheO71Fzw31vhLFurSyg&s"

# ---- Initialize OpenAI clients ----
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key="not-needed")
embed_client = OpenAI(base_url=EMBED_BASE_URL, api_key="not-needed")



## Embedding

In [2]:
# EMBED_URL = EMBED_BASE_URL.replace("/v1", "")
DEFAULT_INSTRUCTION = "Represent the user's input."

def print_truncated_embeddings(data):
    """Helper to print only the first 5 dimensions of each embedding."""
    if "data" in data:
        import copy
        display_data = copy.deepcopy(data)
        for item in display_data["data"]:
            if "embedding" in item:
                original_len = len(item["embedding"])
                item["embedding"] = item["embedding"][:5]
                item["embedding"].append(f"... ({original_len - 5} more values)")
        print(json.dumps(display_data, indent=2))
        print(f"\n[Note] Embedding vectors truncated for readability.")
        print(f"Full dimension size: {original_len}")
    else:
        print(json.dumps(data, indent=2))

def test_embedding_health():
    url = f"{EMBED_BASE_URL}/v1/models"
    resp = requests.get(url)
    print("=== Embedding /v1/models ===")
    print(json.dumps(resp.json(), indent=2))
    print()

def test_embedding_single():
    url = f"{EMBED_BASE_URL}/v1/embeddings"
    payload = {"model": EMBED_MODEL, "input": "Hello world", "encoding_format": "float"}
    resp = requests.post(url, json=payload)
    print("=== Embedding /v1/embeddings (single) ===")
    print(f"Status: {resp.status_code}")
    print_truncated_embeddings(resp.json())
    print()

def test_embedding_text():
    url = f"{EMBED_BASE_URL}/v1/embeddings"
    payload = {
        "model": EMBED_MODEL,
        "input": [
            "The quick brown fox jumps over the lazy dog.",
            "Paris is the capital of France.",
            "Machine learning is a subset of artificial intelligence."
        ],
        "encoding_format": "float"
    }
    resp = requests.post(url, json=payload)
    print("=== Embedding /v1/embeddings (text) ===")
    print(f"Status: {resp.status_code}")
    print_truncated_embeddings(resp.json())
    print()
def test_embedding_multimodal_image_only():
    """Image-only embedding using vLLM's `messages` format."""
    url = f"{EMBED_BASE_URL}/v1/embeddings"
    payload = {
        "model": EMBED_MODEL,
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": f"{DEFAULT_INSTRUCTION}"}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": SAMPLE_IMAGE_URL}},
                    {"type": "text", "text": ""}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": ""}]
            }
        ],
        "encoding_format": "float",
        "continue_final_message": True,
        "add_special_tokens": True
    }
    resp = requests.post(url, json=payload)
    print("=== Embedding /v1/embeddings (image only) ===")
    print(f"Status: {resp.status_code}")
    print_truncated_embeddings(resp.json())
    print()


def test_embedding_multimodal_text_image():
    """Text + image embedding using vLLM's `messages` format."""
    url = f"{EMBED_BASE_URL}/v1/embeddings"
    payload = {
        "model": EMBED_MODEL,
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": f"{DEFAULT_INSTRUCTION}"}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": SAMPLE_IMAGE_URL}},
                    {"type": "text", "text": "A cat sitting in the snow."}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": ""}]
            }
        ],
        "encoding_format": "float",
        "continue_final_message": True,
        "add_special_tokens": True
    }
    resp = requests.post(url, json=payload)
    print("=== Embedding /v1/embeddings (text + image) ===")
    print(f"Status: {resp.status_code}")
    print_truncated_embeddings(resp.json())
    print()

# ----- Run Tests -----
try:
    test_embedding_health()
    test_embedding_single()
    test_embedding_text()
    test_embedding_multimodal_image_only()
    test_embedding_multimodal_text_image()
except Exception as e:
    print(f"Embedding test error: {e}")

=== Embedding /v1/models ===
{
  "object": "list",
  "data": [
    {
      "id": "Qwen3-VL-Embedding-2B",
      "object": "model",
      "created": 1777059680,
      "owned_by": "vllm",
      "root": "/models/Qwen3-VL-Embedding-2B",
      "parent": null,
      "max_model_len": 4096,
      "permission": [
        {
          "id": "modelperm-bb1014cabc41bc9f",
          "object": "model_permission",
          "created": 1777059680,
          "allow_create_engine": false,
          "allow_sampling": true,
          "allow_logprobs": true,
          "allow_search_indices": false,
          "allow_view": true,
          "allow_fine_tuning": false,
          "organization": "*",
          "group": null,
          "is_blocking": false
        }
      ]
    }
  ]
}

=== Embedding /v1/embeddings (single) ===
Status: 200
{
  "id": "embd-8db1e9d5e525074c",
  "object": "list",
  "created": 1777059680,
  "model": "Qwen3-VL-Embedding-2B",
  "data": [
    {
      "index": 0,
      "object": "embeddi

## Reranker

In [3]:
def test_reranker_health():
    url = f"{RERANK_BASE_URL}/v1/models"
    resp = requests.get(url)
    print("=== Reranker /v1/models ===")
    print(json.dumps(resp.json(), indent=2))
    print()

def test_reranker_score():
    url = f"{RERANK_BASE_URL}/score"
    payload = {
        "model": RERANK_MODEL,
        "queries": "What is the capital of France?",
        "documents": [
            "Paris is the capital and most populous city of France.",
            "Berlin is the capital of Germany.",
            "The Eiffel Tower is located in Paris.",
            "Madrid is the capital of Spain."
        ]
    }
    resp = requests.post(url, json=payload)
    print("=== Reranker /score ===")
    print(f"Status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))
    print()

def test_reranker_rerank():
    url = f"{RERANK_BASE_URL}/rerank"
    payload = {
        "model": RERANK_MODEL,
        "query": "What is the capital of France?",
        "documents": [
            "Paris is the capital and most populous city of France.",
            "Berlin is the capital of Germany.",
            "The Eiffel Tower is located in Paris.",
            "Madrid is the capital of Spain."
        ],
        "top_n": 3
    }
    resp = requests.post(url, json=payload)
    print("=== Reranker /rerank ===")
    print(f"Status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))
    print()

def test_reranker_multimodal():
    url = f"{RERANK_BASE_URL}/score"
    payload = {
        "model": RERANK_MODEL,
        "queries": "What animal is shown in the image?",
        "documents": [
            {"content": [{"type": "text", "text": "A golden retriever playing in a park."}]},
            {"content": [{"type": "text", "text": "A tabby cat sitting on a windowsill."}]},
            {"content": [{"type": "image_url", "image_url": {"url": SAMPLE_IMAGE_URL}}]},
            {"content": [
                {"type": "text", "text": "A cute kitten looking at the camera."},
                {"type": "image_url", "image_url": {"url": SAMPLE_IMAGE_URL}}
            ]}
        ]
    }
    resp = requests.post(url, json=payload)
    print("=== Reranker Multimodal /score ===")
    print(f"Status: {resp.status_code}")
    print(json.dumps(resp.json(), indent=2))
    print()

# ----- Run Tests -----
try:
    test_reranker_health()
    test_reranker_score()
    test_reranker_rerank()
    test_reranker_multimodal()
except Exception as e:
    print(f"Reranker test error: {e}")

=== Reranker /v1/models ===
{
  "object": "list",
  "data": [
    {
      "id": "Qwen3-VL-Reranker-2B",
      "object": "model",
      "created": 1777059682,
      "owned_by": "vllm",
      "root": "/models/Qwen3-VL-Reranker-2B",
      "parent": null,
      "max_model_len": 4096,
      "permission": [
        {
          "id": "modelperm-b55fb067c488f049",
          "object": "model_permission",
          "created": 1777059682,
          "allow_create_engine": false,
          "allow_sampling": true,
          "allow_logprobs": true,
          "allow_search_indices": false,
          "allow_view": true,
          "allow_fine_tuning": false,
          "organization": "*",
          "group": null,
          "is_blocking": false
        }
      ]
    }
  ]
}

=== Reranker /score ===
Status: 200
{
  "id": "score-8d55ceb458796abd",
  "object": "list",
  "created": 1777059682,
  "model": "Qwen3-VL-Reranker-2B",
  "data": [
    {
      "index": 0,
      "object": "score",
      "score": 0.602

## LLM

In [4]:
print(f"Testing LLM model: {LLM_MODEL} (Streaming)\n")
response = llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Tell me a short joke about programming."}
    ],
    stream=True
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print()

Testing LLM model: Qwen2.5-VL-7B-Instruct-NVFP4 (Streaming)

Sure! Here's a short programming joke:

Why did the programmer quit his job?

Because he didn't get arrays.
